# Knowledge Bases & RAG

## What is RAG?

RAG solves the hallucination problem by:
1. Converting your documents into vector embeddings
2. Storing embeddings in a vector database
3. Retrieving relevant documents when answering questions
4. Passing retrieved context to the model

```
User Question
    ↓
Vector Embedding
    ↓
Search Vector DB
    ↓
Retrieve Top-K Documents
    ↓
Combine with Prompt
    ↓
Send to Model
    ↓
Grounded Answer
```

## Bedrock Knowledge Bases

Bedrock Knowledge Bases automate the RAG pipeline:

In [ ]:
import boto3
import json

client = boto3.client('bedrock-agent', region_name='us-east-1')

# Create a knowledge base
response = client.create_knowledge_base(
    name='company-docs-kb',
    description='Company policies and procedures',
    roleArn='arn:aws:iam::ACCOUNT:role/BedrockKBRole',
    knowledgeBaseConfiguration={
        'type': 'VECTOR',
        'vectorKnowledgeBaseConfiguration': {
            'embeddingModel': {
                'provider': 'BEDROCK',
                'modelIdentifier': 'amazon.titan-embed-text-v2:0'
            }
        }
    },
    storageConfiguration={
        'type': 'OPENSEARCH_SERVERLESS',
        'opensearchServerlessConfiguration': {
            'collectionArn': 'arn:aws:aoss:us-east-1:ACCOUNT:collection/...'
        }
    }
)

kb_id = response['knowledgeBase']['id']
print(f"Knowledge Base ID: {kb_id}")

## Data Sources

Add documents to your knowledge base:

In [ ]:
# Create a data source
response = client.create_data_source(
    knowledgeBaseId=kb_id,
    name='s3-policies',
    description='Company policies from S3',
    dataSourceConfiguration={
        'type': 'S3',
        's3Configuration': {
            'bucketArn': 'arn:aws:s3:::my-company-docs',
            'inclusionPrefixes': ['policies/'],
            'documentEncodingConfiguration': {
                'encoding': 'UTF-8'
            }
        }
    }
)

data_source_id = response['dataSource']['id']

# Ingest documents
response = client.start_ingestion_job(
    knowledgeBaseId=kb_id,
    dataSourceId=data_source_id
)

print(f"Ingestion Job: {response['ingestionJob']['ingestionJobId']}")

## Vector Embeddings

Embeddings convert text into numerical vectors for similarity search:

In [ ]:
import boto3

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

# Generate embeddings for text
response = bedrock.invoke_model(
    modelId='amazon.titan-embed-text-v2:0',
    body=json.dumps({
        "inputText": "AWS Bedrock is a managed service for foundation models"
    })
)

result = json.loads(response['body'].read())
embedding = result['embedding']  # List of 1024 floats

print(f"Embedding dimension: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")

## Document Chunking

Split large documents into manageable chunks:

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=200):
    """Split text into overlapping chunks"""
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
    return chunks

# Example
document = """
AWS Bedrock is a fully managed service that provides access to foundation models.
It supports multiple models from different providers including Anthropic, Meta, Mistral, and Stability AI.
Bedrock handles scaling, availability, and security automatically.
You can use Bedrock for various tasks including content generation, code assistance, and RAG.
"""

chunks = chunk_text(document, chunk_size=200, overlap=50)
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk[:50]}...")

## Retrieval Patterns

### Basic Retrieval



### Retrieve and Generate

In [ ]:
from boto3 import client as boto3_client

bedrock_agent_runtime = boto3_client('bedrock-agent-runtime', region_name='us-east-1')

# Retrieve documents from knowledge base
response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=kb_id,
    retrievalQuery={
        'text': 'What is the vacation policy?'
    },
    retrievalConfiguration={
        'vectorSearchConfiguration': {
            'numberOfResults': 5,
            'overrideSearchType': 'SEMANTIC'
        }
    }
)

# Process results
for result in response['retrievalResults']:
    print(f"Score: {result['score']}")
    print(f"Content: {result['content']['text'][:200]}...")

In [ ]:
# Retrieve documents and generate answer
response = bedrock_agent_runtime.retrieve_and_generate(
    input={
        'text': 'What is the remote work policy?'
    },
    retrieveAndGenerateConfiguration={
        'type': 'KNOWLEDGE_BASE',
        'knowledgeBaseConfiguration': {
            'knowledgeBaseId': kb_id,
            'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-3-sonnet-20240229-v1:0',
            'retrievalConfiguration': {
                'vectorSearchConfiguration': {
                    'numberOfResults': 5
                }
            }
        }
    }
)

answer = response['output']['text']
print(f"Answer: {answer}")

# See which documents were used
for citation in response['citations']:
    print(f"Source: {citation['generatedResponsePart']}")

## Chunking Strategies

In [ ]:
# Strategy 1: Fixed-size chunks
def fixed_chunks(text, size=500):
    return [text[i:i+size] for i in range(0, len(text), size)]

# Strategy 2: Sentence-based chunks
import re

def sentence_chunks(text, sentences_per_chunk=5):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = ' '.join(sentences[i:i+sentences_per_chunk])
        chunks.append(chunk)
    return chunks

# Strategy 3: Semantic chunks (using embeddings)
def semantic_chunks(text, max_chunk_size=1000):
    """Split at natural boundaries"""
    # Split by paragraphs first
    paragraphs = text.split('\n\n')
    chunks = []
    current_chunk = ""
    
    for para in paragraphs:
        if len(current_chunk) + len(para) < max_chunk_size:
            current_chunk += para + "\n\n"
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = para + "\n\n"
    
    if current_chunk:
        chunks.append(current_chunk)
    
    return chunks

## Best Practices

---

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the main problem that RAG solves?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="0">
      <span>Slow API response times</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="1">
      <span>Model hallucinations by grounding answers in retrieved documents</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="2">
      <span>High token costs</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7384629" value="3">
      <span>Model access limitations</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What is the purpose of document chunking?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8294756" value="0">
      <span>Reduce storage costs</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8294756" value="1">
      <span>Improve document security</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8294756" value="2">
      <span>Split documents into manageable pieces for embedding and retrieval</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8294756" value="3">
      <span>Compress document size</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What does an embedding represent?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9374628" value="0">
      <span>A numerical vector representation of text for similarity search</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9374628" value="1">
      <span>A compressed version of a document</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9374628" value="2">
      <span>A hash of the document content</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9374628" value="3">
      <span>A metadata tag for documents</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is a typical optimal chunk size?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384729" value="0">
      <span>100-200 tokens</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384729" value="1">
      <span>500-1000 tokens</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384729" value="2">
      <span>2000-5000 tokens</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q6384729" value="3">
      <span>Chunk size doesn't matter</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

In [ ]:
# ✅ Good: Metadata for filtering
chunk_with_metadata = {
    "text": "AWS Bedrock pricing...",
    "metadata": {
        "source": "pricing-guide.pdf",
        "date": "2024-01-15",
        "category": "pricing"
    }
}

# ✅ Good: Appropriate chunk size
# Too small: Loses context
# Too large: Retrieves irrelevant info
# Sweet spot: 500-1000 tokens

# ✅ Good: Overlap between chunks
# Prevents losing information at boundaries
# Typical overlap: 10-20% of chunk size

# ✅ Good: Regular re-indexing
# Update knowledge base when documents change
# Monitor ingestion job status